# DATA PREP FOR HOW AI IS SHAPING THE WORKFORCE

This notebook is for exploratory puruposes. If regular data refresh is required, move this into a structured scripted.

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import json

## Load in BLS AI exposure data

In [2]:
bls_ai = pd.read_excel('ai-exposure-categories.xlsx', skiprows=1)
bls_ai = bls_ai.dropna(subset=['2025 National Employment Matrix code'])
# bls_ai.info(show_counts=True)
# cleaned_bls_ai = bls_ai.where(pd.notna(bls_ai), None).to_dict(orient='records')
occupation_df = pd.read_excel('occupation.xlsx', sheet_name='Table 1.10', skiprows=1)
occupation_df = occupation_df.dropna(subset=['2025 National Employment Matrix code'])
occupation_df = (occupation_df[
    [
        '2025 National Employment Matrix code',
        'Labor force exit rate, 2025–35 annual average',
        'Occupational transfer rate, 2025–35 annual average',
        'Total occupational separations rate, 2025–35 annual average',
        'Labor force exits, 2025–35 annual average',
        'Occupational transfers, 2025–35 annual average',
        'Total occupational separations, 2025–35 annual average',
        'Occupational openings, 2025–35 annual average'
    ]
])
# occupation_df.info(show_counts=True)

joined_df = pd.merge(bls_ai, occupation_df, how='left', on='2025 National Employment Matrix code')


joined_df = joined_df.rename(columns={
    '2025 National Employment Matrix code': 'SOC code',
    '2025 National Employment Matrix title': 'Occupation title',
    'Employment change, percent, 2025–35': 'Employment change, percent',
    'Labor force exit rate, 2025–35 annual average': 'Labor force exit rate, annual average',
    'Occupational transfer rate, 2025–35 annual average': 'Occupational transfer rate, annual average',
    'Total occupational separations rate, 2025–35 annual average': 'Total occupational separations rate, annual average',
    'Labor force exits, 2025–35 annual average': 'Labor force exits, annual average',
    'Occupational transfers, 2025–35 annual average': 'Occupational transfers, annual average',
    'Total occupational separations, 2025–35 annual average': 'Total occupational separations, annual average',
    'Occupational openings, 2025–35 annual average': 'Occupational openings, annual average'
})
joined_df['Industry code'] = joined_df['SOC code'].str.split('-',n=1).str[0]
joined_df.info(show_counts=True)
# cleaned_occupation_df = occupation_df.where(pd.notna(occupation_df), None).to_dict(orient='records')

<class 'pandas.DataFrame'>
RangeIndex: 831 entries, 0 to 830
Data columns (total 21 columns):
 #   Column                                                                     Non-Null Count  Dtype  
---  ------                                                                     --------------  -----  
 0   Occupation title                                                           831 non-null    str    
 1   SOC code                                                                   831 non-null    str    
 2   Employment, 2025                                                           831 non-null    float64
 3   Employment, 2035                                                           831 non-null    float64
 4   Employment change, numeric, 2025–35                                        831 non-null    float64
 5   Employment change, percent                                                 831 non-null    float64
 6   Percent self-employed, 2025                                          

In [3]:
industry_cats = pd.read_excel('occupation.xlsx', sheet_name='Table 1.10', skiprows=1)
industry_cats = industry_cats[industry_cats['2025 National Employment Matrix code'].str.endswith('-0000')]
industry_cats = industry_cats[['2025 National Employment Matrix code', '2025 National Employment Matrix title']]
industry_cats = industry_cats.rename(columns={
    '2025 National Employment Matrix code': 'Industry code', '2025 National Employment Matrix title': 'Industry title'
})
industry_cats['Industry code'] = industry_cats['Industry code'].str.replace('-0000','', regex=False)

joined_df = pd.merge(joined_df, industry_cats, how='left', on='Industry code')
joined_df.head(2)
cleaned_df = (
    joined_df
    .replace({pd.NA: None, np.nan: None})
    .where(pd.notna(joined_df), None)
    .to_dict(orient='records')
    )

In [4]:
trends = pd.read_excel('Sector.xlsx', sheet_name='Response Estimates')
trends['Sector'] = pd.to_numeric(trends['Sector'], errors='coerce')
industry_cats['Industry code'] = pd.to_numeric(industry_cats['Industry code'], errors='coerce')
trends = pd.merge(trends, industry_cats, how='left', left_on='Sector', right_on='Industry code')

ai_trends = trends[trends['Question ID'].isin([7.0, 28.0])]
ai_trends = ai_trends[['Sector', 'Question ID', 'Question', 'Answer ID', 'Answer', '202622',
       '202621', '202620', '202619', '202618', '202617', '202616', '202615',
       '202614', '202613', '202612', '202611', '202610', '202609', '202608',
       '202607', '202606', '202605', '202604', '202603', '202602', '202601',
       '202526', '202525', '202524','Industry title']]
ai_melt = ai_trends.melt(
    id_vars=['Sector', 'Question ID', 'Question', 'Answer ID', 'Answer', 'Industry title'],
    value_vars=[col for col in ai_trends.columns if col.startswith('20')],
    var_name='Response Period',
    value_name='Response Rate'
)
ai_melt['Response Period'] = ai_melt['Response Period'].astype("string").str.zfill(6)
ai_melt['sample_year'] = ai_melt['Response Period'].str[:4].astype(int)
ai_melt['period'] = ai_melt['Response Period'].str[4:].astype(int)

period1_start = {
    2025: pd.Timestamp("2024-12-30"),
    2026: pd.Timestamp("2025-12-29")
}

ai_melt['period_start_date'] = (
    ai_melt.sample_year.map(period1_start)
    + pd.to_timedelta((ai_melt.period - 1) * 14, unit='D')
)
ai_melt['period_end_date'] = ai_melt['period_start_date'] + pd.Timedelta(days=13)

response_text = ai_melt['Response Rate'].astype('string').str.strip()
ai_melt['has_poor_stats'] = response_text.eq('S').fillna(False)
response_rate = pd.to_numeric(
    response_text.str.replace('%', '', regex=False), errors='coerce'
)
group_cols = ['Sector', 'Question ID', 'Response Period']
group_keys = [ai_melt[col] for col in group_cols]
missing_count = response_rate.isna().groupby(
    group_keys, dropna=False
).transform('sum')
known_total = response_rate.groupby(
    group_keys, dropna=False
).transform('sum', min_count=1)
remainder = 100 - known_total
can_impute = (
    response_rate.isna()
    & missing_count.eq(1)
    & remainder.between(0, 100)
)
ai_melt['rate'] = response_rate.mask(can_impute, remainder)

q7_rates = ai_melt.loc[
    ai_melt['Question ID'].eq(7),
    ['Sector', 'Answer', 'period_end_date', 'rate']
].copy()
q7_rates['_row_index'] = q7_rates.index
q7_rates['six_months_prior'] = (
    q7_rates['period_end_date'] - pd.DateOffset(months=6)
)

q28_periods = (
    ai_melt.loc[
        ai_melt['Question ID'].eq(28),
        ['Response Period', 'period_start_date', 'period_end_date']
    ]
    .drop_duplicates()
    .rename(columns={
            'Response Period': 'prior_response_period',
            'period_start_date': 'prior_period_start_date',
            'period_end_date': 'prior_period_end_date'
        })
        .sort_values('prior_period_start_date')
    )
q7_rates = pd.merge_asof(
        q7_rates.sort_values('six_months_prior'),
        q28_periods,
        left_on='six_months_prior',
        right_on='prior_period_start_date',
        direction='backward'
    )
prior_period_matches = q7_rates['six_months_prior'].between(
        q7_rates['prior_period_start_date'], q7_rates['prior_period_end_date']
    )
q7_rates.loc[~prior_period_matches, 'prior_response_period'] = pd.NA
    
q28_rates = (
        ai_melt.loc[
            ai_melt['Question ID'].eq(28),
            ['Sector', 'Answer', 'Response Period', 'rate']
        ]
        .rename(columns={
            'Response Period': 'prior_response_period',
            'rate': 'prior_rate'
        })
    )
q7_rates = q7_rates.merge(
        q28_rates,
        on=['Sector', 'Answer', 'prior_response_period'],
        how='left',
        validate='many_to_one'
    )
ai_melt['prior_rate'] = np.nan
ai_melt.loc[q7_rates['_row_index'], 'prior_rate'] = (
        q7_rates['prior_rate'].to_numpy()
    )
q7_rates['rate_difference_6m'] = q7_rates['rate'] - q7_rates['prior_rate']
ai_melt['rate_difference_6m'] = np.nan
ai_melt.loc[q7_rates['_row_index'], 'rate_difference_6m'] = (
        q7_rates['rate_difference_6m'].to_numpy()
    )
q7_same_period = ai_melt.loc[
        ai_melt['Question ID'].eq(7),
        ['Sector', 'Answer', 'Response Period', 'rate']
    ].copy()
q7_same_period['_row_index'] = q7_same_period.index
q28_same_period = ai_melt.loc[
        ai_melt['Question ID'].eq(28),
        ['Sector', 'Answer', 'Response Period', 'rate']
    ].rename(columns={'rate': 'future_rate'})
q7_same_period = q7_same_period.merge(
        q28_same_period,
        on=['Sector', 'Answer', 'Response Period'],
        how='left',
        validate='one_to_one'
    )
ai_melt['future_rate'] = np.nan
ai_melt.loc[q7_same_period['_row_index'], 'future_rate'] = (
        q7_same_period['future_rate'].to_numpy()
    )
q7_same_period['rate_difference_future_6m'] = (
        q7_same_period['future_rate'] - q7_same_period['rate'] 
    )
ai_melt['rate_difference_future_6m'] = np.nan
ai_melt.loc[q7_same_period['_row_index'], 'rate_difference_future_6m'] = (
        q7_same_period['rate_difference_future_6m'].to_numpy()
        )
counts = (
    joined_df
    .dropna(subset=["Industry title", "Relative AI exposure"])
    .groupby(["Industry title", "Relative AI exposure"])
    .size()
    .rename("occupation_count")
    .reset_index()
)

industry_exposure = counts.loc[
    counts.groupby("Industry title")["occupation_count"].idxmax()
].rename(columns={"Relative AI exposure": "General AI exposure"})

industry_exposure['Industry title'] = industry_exposure['Industry title'].str.strip()

In [5]:
ai_melt[['period_start_date', 'period_end_date']] = ai_melt[['period_start_date', 'period_end_date']].astype('string')
ai_melt['Industry title'] = ai_melt['Industry title'].str.strip()
ai_melt = pd.merge(ai_melt, industry_exposure, how='left', on='Industry title')

trends_cleaned = (
    ai_melt
    .replace({pd.NA: None, np.nan: None})
    .where(pd.notna(ai_melt), None)
    .to_dict(orient='records')
)

In [6]:
payload = {
    "raw_data": cleaned_df,
    "trends": trends_cleaned
}

with open('../../public/data/bls_ai_exposure.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, allow_nan=False)

In [ ]:
ai_exposure_rank_order = ['Very high', 'High', 'Moderate', 'Low']

joined_df['Relative AI exposure'] = pd.Categorical(joined_df['Relative AI exposure'], categories=ai_exposure_rank_order, ordered=True)
averages = joined_df.groupby('Relative AI exposure')['Employment change, percent'].agg(['mean', 'median']).reset_index()

In [ ]:
fig = px.scatter(
    joined_df,
    y='Employment change, percent',
    x='Total occupational separations rate, annual average',
    height=600,
    width=900,
    hover_data=['Occupation title'],
    title='AI Exposure vs Job Forecasted Change (%)',
    color='Relative AI exposure',
    category_orders={'Relative AI exposure': ai_exposure_rank_order},
)

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.7,
        line=dict(width=1, color='DarkSlateGrey')
    )
)

separation_median = joined_df['Total occupational separations rate, annual average'].median()

fig.add_vline(
    x=separation_median,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Median separation rate: {separation_median:.2f}",
    annotation_position="top left"
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

# fig.add_trace(
#     go.Scatter(
#         x=averages['mean'],
#         y=averages['Relative AI exposure'],
#         mode='text+markers',
#         marker=dict(symbol='diamond', size=10),
#         name='Average',
#     )
# )

# fig.add_trace(
#     go.Scatter(
#         x=averages['median'],
#         y=averages['Relative AI exposure'],
#         mode='text+markers',
#         marker=dict(symbol='diamond', size=10),
#         name='Median',
#     )
# )


fig.update_layout(
    coloraxis_showscale=False,
    template='plotly_white',
)
fig.show()